# OpenAPI y validacion

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/19_diseno_api/code/01_openapi_y_validacion.ipynb)


## 1. Setup

Instalamos dependencias ligeras para cargar YAML y validar payloads.

In [ ]:
%pip install pyyaml jsonschema

In [ ]:
import json
import yaml
from jsonschema import validate, ValidationError

## 2. Contrato minimo del chatbot

Definimos un OpenAPI pequeno como string para poder inspeccionarlo localmente.

In [ ]:
openapi_yaml = '''
openapi: 3.1.0
info:
  title: Chatbot API
  version: 1.0.0
paths:
  /v1/chat:
    post:
      summary: Crear respuesta del chatbot
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/ChatRequest'
      responses:
        '200':
          description: Respuesta correcta
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/ChatResponse'
components:
  schemas:
    Message:
      type: object
      required: [role, content]
      properties:
        role:
          type: string
          enum: [user, assistant, system]
        content:
          type: string
    ChatRequest:
      type: object
      required: [model, messages]
      properties:
        model:
          type: string
        temperature:
          type: number
          minimum: 0
          maximum: 2
        messages:
          type: array
          minItems: 1
          items:
            $ref: '#/components/schemas/Message'
    ChatResponse:
      type: object
      required: [id, output_text]
      properties:
        id:
          type: string
        output_text:
          type: string
        usage:
          type: object
          properties:
            input_tokens:
              type: integer
            output_tokens:
              type: integer
'''

spec = yaml.safe_load(openapi_yaml)
print(spec['info'])

In [ ]:
chat_request_schema = spec['components']['schemas']['ChatRequest']
message_schema = spec['components']['schemas']['Message']

chat_request_schema['properties']['messages']['items'] = message_schema
print(json.dumps(chat_request_schema, indent=2))

## 3. Payload valido vs invalido

In [ ]:
payload_valido = {
    'model': 'chat-mini',
    'temperature': 0.4,
    'messages': [
        {'role': 'user', 'content': 'Explica que es un contrato API'}
    ]
}

payload_invalido = {
    'model': 'chat-mini',
    'temperature': 3.5,
    'messages': []
}

validate(instance=payload_valido, schema=chat_request_schema)
print('Payload valido: OK')

try:
    validate(instance=payload_invalido, schema=chat_request_schema)
except ValidationError as e:
    print('Payload invalido detectado:')
    print('-', e.message)

## 4. Que ambiguedades resolvimos?

OpenAPI nos obligo a decidir:

- que campos son obligatorios
- que rangos son validos
- que forma tienen las listas
- que shape promete la respuesta